# PINNacle Chain Evaluator — Kaggle Runner

Loads the best optimizer chain from `best_optimizer_chains.json`,
runs it **N_SEEDS times** with different seeds (N_PROCESSES in parallel),
and appends results to the shared CSV on Hugging Face Hub.

**Set SMOKE_TEST = True for a quick end-to-end check before a full run.**

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
GITHUB_REPO = "https://github.com/florentiner/PINNacle.git"
BRANCH      = "optuna_exp"

# PDE and chain type
PDE_NAME   = "wave1d"   # must match a key in best_optimizer_chains.json
VALUE_TYPE = "fixed"    # "fixed" or "continuous"

# Parallelism
N_PROCESSES = 2   # workers running simultaneously
N_SEEDS     = 10  # total evaluation seeds
SEED_BASE   = 42  # seeds = SEED_BASE .. SEED_BASE+N_SEEDS-1

# Network
HIDDEN_LAYERS = "100*5"
DISPLAY_EVERY = 100

# HF Hub — shared results CSV (public repo, write needs HF_TOKEN_WRITE secret)
HF_REPO_ID  = "danil-e/pinnacle-optuna-db"
HF_CSV_FILE = "csv_seed/chain_eval_results.csv"

# ── Test / Smoke ───────────────────────────────────────────────────────────────
# SMOKE_TEST=True  → random 5-stage chain, TEST_EPOCHS epochs/stage, 2 seeds only
# TEST_EPOCHS=None → use epochs from chain JSON (full run)
# TEST_EPOCHS=2    → cap every stage to 2 epochs (fast end-to-end check)
SMOKE_TEST  = False
TEST_EPOCHS = None
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
import torch, os
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    for _i in range(torch.cuda.device_count()):
        _p = torch.cuda.get_device_properties(_i)
        print(f"  GPU {_i}: {_p.name}  sm_{_p.major}{_p.minor}")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    print("  MPS (Apple Silicon) available.")
else:
    print("  No GPU — running on CPU.")

In [ ]:
import subprocess, os
if not os.path.exists("PINNacle") and not os.path.exists("experiments"):
    subprocess.run(["git", "clone", "-b", BRANCH, "--single-branch", GITHUB_REPO, "PINNacle"], check=True)
else:
    print("Repo already present.")

In [ ]:
if os.path.exists("PINNacle") and not os.path.exists("experiments"):
    os.chdir("PINNacle")
print("Working directory:", os.getcwd())

In [ ]:
!pip install huggingface_hub pandas -q

In [ ]:
import sys
sys.path.insert(0, os.getcwd())
os.environ["DDEBACKEND"] = "pytorch"
import deepxde as dde
print("DeepXDE:", dde.__version__)

In [ ]:
import json, random
from huggingface_hub import hf_hub_download as _hf_dl

KEY = f"{PDE_NAME}_{VALUE_TYPE}"

# Download best_optimizer_chains.json from HF Hub (public, no token needed)
_chains_local = _hf_dl(
    repo_id=HF_REPO_ID, filename="best_optimizer_chains.json",
    repo_type="dataset", token=None,
    local_dir="hf_cache", force_download=True,
)
with open(_chains_local) as f:
    all_chains = json.load(f)

_smoke_epochs = TEST_EPOCHS if TEST_EPOCHS is not None else 3

if SMOKE_TEST:
    _OPTS     = ["Adam", "LBFGS", "PSO"]
    _LR_RANGE = {"Adam": (1e-4, 1e-2), "LBFGS": (0.1, 1.0), "PSO": (0.0, 1e-3)}
    chain = []
    for _ in range(5):
        opt = random.choice(_OPTS)
        lo, hi = _LR_RANGE[opt]
        chain.append({"optimizer": opt, "lr": round(random.uniform(lo, hi), 6), "epochs": _smoke_epochs})
    _run_seeds = list(range(SEED_BASE, SEED_BASE + 2))   # 2 seeds for smoke test
    print(f"SMOKE TEST: random chain, epochs={_smoke_epochs}, 2 seeds")
else:
    if KEY not in all_chains:
        avail = sorted(all_chains.keys())
        raise KeyError(f"Key '{KEY}' not in best_optimizer_chains.json.\nAvailable: {avail}")
    chain = all_chains[KEY]
    _run_seeds = list(range(SEED_BASE, SEED_BASE + N_SEEDS))
    print(f"Loaded chain for '{KEY}' ({len(chain)} stages) from HF Hub")

# Cap epochs for test runs
if TEST_EPOCHS is not None:
    chain = [dict(s, epochs=TEST_EPOCHS) for s in chain]
    print(f"TEST_EPOCHS={TEST_EPOCHS}: all stage epochs capped to {TEST_EPOCHS}")

# Write chain to temp file for workers
_CHAIN_FILE = "chain_to_run.json"
with open(_CHAIN_FILE, "w") as f:
    json.dump(chain, f)

print("\nChain:")
for i, s in enumerate(chain):
    print(f"  Stage {i}: {s['optimizer']:5s}  lr={s['lr']:.4g}  epochs={s['epochs']}")
print(f"\nSeeds to run: {_run_seeds}")

In [ ]:
import pandas as pd, os

_hf_write = None
try:
    from kaggle_secrets import UserSecretsClient as _USC
    try: _hf_write = _USC().get_secret("HF_TOKEN_WRITE")
    except Exception: pass
except Exception: pass
_hf_write = _hf_write or os.environ.get("HF_TOKEN_WRITE")

if _hf_write:
    print("HF_TOKEN_WRITE loaded — results will be uploaded after run.")
else:
    print("INFO: HF_TOKEN_WRITE not set — results saved locally only.")

_existing_df = None
try:
    from huggingface_hub import hf_hub_download as _hf_dl
    os.makedirs("hf_cache", exist_ok=True)
    _csv_local = _hf_dl(
        repo_id=HF_REPO_ID, filename=HF_CSV_FILE,
        repo_type="dataset", token=None,
        local_dir="hf_cache", force_download=True,
    )
    _existing_df = pd.read_csv(_csv_local)
    print(f"Downloaded existing CSV from HF Hub: {len(_existing_df)} rows")
except Exception as _e:
    _ename = type(_e).__name__
    if "404" in str(_e) or "not found" in str(_e).lower() or "EntryNotFound" in _ename:
        print("No existing CSV on HF Hub — will create new file.")
    else:
        print(f"WARNING: Could not download CSV: {_e}")

In [ ]:
import subprocess, time, sys, os

WORKER_SCRIPT = "experiments/optuna_multi_pde/chain_eval_worker.py"
RESULTS_DIR   = "eval_results"
PYTHON_BIN    = sys.executable
os.makedirs(RESULTS_DIR, exist_ok=True)

# GPU filtering (skip sm<70: P100 etc. unsupported by PyTorch 2.x)
compatible_gpus = []
for _gid in range(torch.cuda.device_count()):
    _props = torch.cuda.get_device_properties(_gid)
    if _props.major * 10 + _props.minor >= 70:
        compatible_gpus.append(_gid)
    else:
        print(f"Skipping GPU {_gid} ({_props.name} sm_{_props.major}{_props.minor}): requires sm>=70")
n_gpus  = len(compatible_gpus)
use_mps = (n_gpus == 0 and getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())

print(f"GPUs usable: {compatible_gpus or ('mps' if use_mps else 'none — CPU')}")
print(f"Launching {len(_run_seeds)} seeds, {N_PROCESSES} parallel\n")

active = []   # list of (Popen, seed)

def _launch_next(i, seed):
    result_json = os.path.join(RESULTS_DIR, f"result_seed_{seed}.json")
    env = os.environ.copy()
    env["DDEBACKEND"] = "pytorch"
    if use_mps:
        env["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"
    elif n_gpus > 0:
        env["CUDA_VISIBLE_DEVICES"] = str(compatible_gpus[i % n_gpus])
    else:
        env["CUDA_VISIBLE_DEVICES"] = ""
    cmd = [
        PYTHON_BIN, WORKER_SCRIPT,
        "--pde-name",      PDE_NAME,
        "--chain-json",    _CHAIN_FILE,
        "--seed",          str(seed),
        "--result-json",   result_json,
        "--display-every", str(DISPLAY_EVERY),
        "--hidden-layers", HIDDEN_LAYERS,
    ]
    p = subprocess.Popen(cmd, env=env)
    print(f"  Started seed {seed} (PID {p.pid})")
    return p

for i, seed in enumerate(_run_seeds):
    # Wait if all slots occupied
    while len(active) >= N_PROCESSES:
        still_active = []
        for p, s in active:
            if p.poll() is not None:
                print(f"  Seed {s} finished (exit {p.returncode})")
            else:
                still_active.append((p, s))
        active = still_active
        if len(active) >= N_PROCESSES:
            time.sleep(5)
    p = _launch_next(i, seed)
    active.append((p, seed))

# Drain remaining
for p, s in active:
    p.wait()
    print(f"  Seed {s} finished (exit {p.returncode})")

print("\nAll seeds done.")

In [ ]:
import json, math, pandas as pd, os
from datetime import datetime

rows = []
for seed in _run_seeds:
    rpath = os.path.join(RESULTS_DIR, f"result_seed_{seed}.json")
    if not os.path.exists(rpath):
        print(f"WARNING: missing result for seed {seed}")
        continue
    with open(rpath) as f:
        r = json.load(f)
    mse    = r.get("mse",    float("nan"))   # operator MSE  = rmse²
    brmse  = r.get("brmse",  float("nan"))   # boundary RMSE
    l2re   = r.get("l2re",   float("nan"))
    bc_l2re = r.get("bc_l2re", float("nan"))

    mse_op  = mse
    mse_bnd = brmse ** 2 if math.isfinite(brmse) else float("nan")
    mse_tot = mse_op + mse_bnd if (math.isfinite(mse_op) and math.isfinite(mse_bnd)) else float("nan")
    l2re_op  = l2re
    l2re_bnd = bc_l2re
    l2re_tot = l2re_op + l2re_bnd if (math.isfinite(l2re_op) and math.isfinite(l2re_bnd)) else float("nan")

    rows.append({
        "run_timestamp": datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
        "pde_name":   PDE_NAME,
        "value_type": VALUE_TYPE,
        "smoke_test": SMOKE_TEST,
        "chain_key":  KEY if not SMOKE_TEST else "smoke_test",
        "seed":       seed,
        # MSE columns
        "mse_op":    mse_op,
        "mse_bnd":   mse_bnd,
        "mse_total": mse_tot,
        # L2RE columns
        "l2re_op":    l2re_op,
        "l2re_bnd":   l2re_bnd,
        "l2re_total": l2re_tot,
        "elapsed_s":  r.get("elapsed_s"),
        "chain_json": json.dumps(chain),
    })

new_df = pd.DataFrame(rows)
print(f"New results ({len(new_df)} seeds):")
display(new_df[["seed", "mse_op", "mse_bnd", "mse_total", "l2re_op", "l2re_bnd", "l2re_total", "elapsed_s"]])

# Summary stats
print(f"\n{'='*60}")
print(f"Summary: {PDE_NAME}_{VALUE_TYPE}  ({len(new_df)} seeds)")
print(f"{'='*60}")
for col in ["mse_op", "mse_bnd", "mse_total", "l2re_op", "l2re_bnd", "l2re_total"]:
    v = new_df[col].replace([float("inf"), float("-inf")], float("nan")).dropna()
    if len(v):
        print(f"  {col:12s}: mean={v.mean():.4e}  std={v.std():.4e}  "
              f"min={v.min():.4e}  max={v.max():.4e}")

# Append to existing CSV
combined_df = pd.concat([_existing_df, new_df], ignore_index=True) if _existing_df is not None else new_df
os.makedirs(os.path.dirname(HF_CSV_FILE), exist_ok=True)
combined_df.to_csv(HF_CSV_FILE, index=False)
print(f"\nTotal rows in combined CSV: {len(combined_df)}")

# Upload to HF Hub
if _hf_write:
    try:
        from huggingface_hub import upload_file as _hf_up
        _hf_up(
            path_or_fileobj=HF_CSV_FILE,
            path_in_repo=HF_CSV_FILE,
            repo_id=HF_REPO_ID,
            repo_type="dataset",
            token=_hf_write,
            commit_message=f"add {len(new_df)} {PDE_NAME}_{VALUE_TYPE} results (smoke={SMOKE_TEST})",
        )
        print(f"Uploaded to https://huggingface.co/datasets/{HF_REPO_ID}")
    except Exception as _e:
        print(f"WARNING: HF upload failed: {_e}")
else:
    print(f"Saved locally: {HF_CSV_FILE}")